# LoRA Fine-tuning for Satellite Inpainting
Fine-tunes SD2 inpainting on the RSICD satellite image-caption dataset using LoRA.
Trains only small adapter weights (~3MB) on top of the frozen base model.

**Hardware:** Tested on RTX 3080 10GB. Training ~1-2 hours for 3 epochs.

In [ ]:
!uv add peft accelerate datasets -q

In [1]:
# --- All hyperparameters in one place ---
CFG = dict(
    base_model     = "sd2-community/stable-diffusion-2-inpainting",
    dataset        = "arampacha/rsicd",
    output_dir     = "./lora-satellite-inpaint",

    # LoRA
    lora_rank      = 4,
    lora_alpha     = 4,
    lora_dropout   = 0.05,

    # Training
    image_size     = 512,
    batch_size     = 1,
    grad_accum     = 4,          # effective batch = 4
    lr             = 1e-4,
    num_epochs     = 3,
    max_steps      = 2000,       # stops early if hit before epochs
    save_every     = 500,

    # Mask (random rectangles during training)
    mask_min_frac  = 0.1,        # min 10% of image masked
    mask_max_frac  = 0.5,        # max 50% of image masked
)


In [6]:
import os, random, torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from diffusers import StableDiffusionInpaintPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, UNet2DConditionModel
from tqdm.auto import tqdm
final_path = os.path.join(CFG["output_dir"], "lora-final")
device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(CFG["output_dir"], exist_ok=True)
print(f"Device: {device} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda | VRAM: 10.7 GB


In [10]:
raw = load_dataset(CFG["dataset"], split="train")
print(f"Dataset size: {len(raw)} images")
print(f"Columns: {raw.column_names}")
print(f"Sample caption: {raw[0]['captions'][0]}")

Dataset size: 8734 images
Columns: ['filename', 'captions', 'image']
Sample caption: Many aircraft are parked next to a long building in an airport.


In [11]:
def random_rect_mask(h, w, min_frac, max_frac):
    """Returns a binary mask (1=inpaint) with a random rectangle."""
    mask = np.zeros((h, w), dtype=np.float32)
    area = h * w
    target = random.uniform(min_frac, max_frac) * area
    rh = random.randint(int(h * 0.1), int(h * 0.7))
    rw = int(target / rh)
    rw = max(1, min(rw, w))
    y0 = random.randint(0, h - rh)
    x0 = random.randint(0, w - rw)
    mask[y0:y0+rh, x0:x0+rw] = 1.0
    return mask


class RSICDInpaintDataset(Dataset):
    def __init__(self, hf_dataset, image_size, mask_min_frac, mask_max_frac):
        self.data = hf_dataset
        self.size = image_size
        self.min_f = mask_min_frac
        self.max_f = mask_max_frac
        self.tf = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),  # [-1, 1]
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img = item["image"]
        if isinstance(img, dict):          # HF image format
            img = Image.fromarray(img["bytes"]) if isinstance(img.get("bytes"), bytes) else img
        img = img.convert("RGB")
        caption = random.choice(item["captions"])  # pick one caption randomly

        image_t = self.tf(img)             # [3, H, W] in [-1,1]
        mask = random_rect_mask(self.size, self.size, self.min_f, self.max_f)
        mask_t = torch.from_numpy(mask).unsqueeze(0)  # [1, H, W]

        # Masked image: keep unmasked pixels, zero out masked region
        masked_image_t = image_t * (1 - mask_t)

        return {"image": image_t, "mask": mask_t, "masked_image": masked_image_t, "caption": caption}


dataset = RSICDInpaintDataset(
    raw, CFG["image_size"], CFG["mask_min_frac"], CFG["mask_max_frac"]
)
loader = DataLoader(dataset, batch_size=CFG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
print(f"Batches per epoch: {len(loader)}")

Batches per epoch: 8734


In [12]:
# Load frozen components
tokenizer    = CLIPTokenizer.from_pretrained(CFG["base_model"], subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(CFG["base_model"], subfolder="text_encoder", torch_dtype=torch.float16).to(device)
vae          = AutoencoderKL.from_pretrained(CFG["base_model"], subfolder="vae", torch_dtype=torch.float16).to(device)
noise_sched  = DDPMScheduler.from_pretrained(CFG["base_model"], subfolder="scheduler")

text_encoder.requires_grad_(False)
vae.requires_grad_(False)

# Load UNet and apply LoRA
unet = UNet2DConditionModel.from_pretrained(CFG["base_model"], subfolder="unet", torch_dtype=torch.float32).to(device)

lora_cfg = LoraConfig(
    r=CFG["lora_rank"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],  # attention projections only
)
unet = get_peft_model(unet, lora_cfg)
unet.print_trainable_parameters()

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: sd2-community/stable-diffusion-2-inpainting
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/sikk1/src/inf-3600-satellite-inpaint/notebooks/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


trainable params: 829,952 || all params: 866,755,076 || trainable%: 0.0958


In [ ]:
optimizer = torch.optim.AdamW(unet.parameters(), lr=CFG["lr"])
scaler    = torch.cuda.amp.GradScaler()

global_step = 0
unet.train()

for epoch in range(CFG["num_epochs"]):
    epoch_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc=f"Epoch {epoch+1}")):
        images        = batch["image"].to(device, dtype=torch.float16)
        masks         = batch["mask"].to(device, dtype=torch.float16)
        masked_images = batch["masked_image"].to(device, dtype=torch.float16)
        captions      = batch["caption"]

        with torch.no_grad():
            # Encode images and masked images to latent space
            latents        = vae.encode(images).latent_dist.sample() * vae.config.scaling_factor
            masked_latents = vae.encode(masked_images).latent_dist.sample() * vae.config.scaling_factor

            # Downsample mask to latent resolution (H/8, W/8)
            mask_latent = F.interpolate(masks, size=latents.shape[-2:], mode="nearest")

            # Encode captions
            tok = tokenizer(captions, padding="max_length", max_length=tokenizer.model_max_length,
                            truncation=True, return_tensors="pt").to(device)
            encoder_hidden = text_encoder(tok.input_ids)[0]

        # Add noise to latents
        noise      = torch.randn_like(latents)
        timesteps  = torch.randint(0, noise_sched.config.num_train_timesteps,
                                   (latents.shape[0],), device=device).long()
        noisy_lat  = noise_sched.add_noise(latents, noise, timesteps)

        # SD inpainting UNet expects 9 channels: [noisy, mask, masked_image]
        unet_input = torch.cat([noisy_lat, mask_latent, masked_latents], dim=1)

        with torch.cuda.amp.autocast():
            noise_pred = unet(unet_input, timesteps, encoder_hidden_states=encoder_hidden).sample
            # Loss weighted by mask — focus learning on the inpainted region
            loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="none")
            loss = (loss * mask_latent).mean() / CFG["grad_accum"]

        scaler.scale(loss).backward()
        epoch_loss += loss.item() * CFG["grad_accum"]

        if (step + 1) % CFG["grad_accum"] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            global_step += 1

            if global_step % CFG["save_every"] == 0:
                ckpt = os.path.join(CFG["output_dir"], f"checkpoint-{global_step}")
                unet.save_pretrained(ckpt)
                print(f"  Saved checkpoint → {ckpt}")

            if global_step >= CFG["max_steps"]:
                break

    avg = epoch_loss / len(loader)
    print(f"Epoch {epoch+1} — avg loss: {avg:.4f}")
    if global_step >= CFG["max_steps"]:
        break

print("Training complete!")

/tmp/ipykernel_417076/18700419.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()


Epoch 1:   0%|          | 0/8734 [00:00<?, ?it/s]

/tmp/ipykernel_417076/18700419.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Saved checkpoint → ./lora-satellite-inpaint/checkpoint-500


In [ ]:
# Save final LoRA weights
unet.save_pretrained(final_path)
print(f"LoRA weights saved to {final_path}")
print(f"Size: {sum(os.path.getsize(os.path.join(final_path, f)) for f in os.listdir(final_path)) / 1e6:.1f} MB")

In [7]:
# Load base pipeline and inject fine-tuned LoRA UNet for inference
from peft import PeftModel

base_unet = UNet2DConditionModel.from_pretrained(CFG["base_model"], subfolder="unet", torch_dtype=torch.float16)
lora_unet = PeftModel.from_pretrained(base_unet, final_path).to(device)

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    CFG["base_model"],
    unet=lora_unet,
    torch_dtype=torch.float16,
).to(device)

print("Pipeline ready with fine-tuned LoRA UNet")

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /home/sikk1/.cache/huggingface/hub/models--sd2-community--stable-diffusion-2-inpainting/snapshots/5f74973cbb64c8568780732c17f43eb269d63a0d/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Expected types for unet: (<class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>,), got <class 'peft.peft_model.PeftModel'>.


Pipeline ready with fine-tuned LoRA UNet


In [8]:
import ipywidgets as widgets
from IPython.display import display

# Quick test on a sample from the dataset
sample      = dataset[0]
orig_tensor = sample["image"]
mask_tensor = sample["mask"]
caption     = sample["caption"]

# Convert back to PIL for the pipeline
to_pil = transforms.ToPILImage()
orig_pil = to_pil((orig_tensor * 0.5 + 0.5).clamp(0, 1))
mask_pil = to_pil(mask_tensor)

result = pipe(
    prompt=caption,
    image=orig_pil,
    mask_image=mask_pil,
    num_inference_steps=30,
    guidance_scale=12.0,
).images[0]

def pil_bytes(img):
    import io
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

print(f"Prompt: {caption}")
display(widgets.HBox([
    widgets.VBox([widgets.Label("Original"),      widgets.Image(value=pil_bytes(orig_pil), format="png")]),
    widgets.VBox([widgets.Label("Mask"),           widgets.Image(value=pil_bytes(mask_pil), format="png")]),
    widgets.VBox([widgets.Label("LoRA Inpainted"), widgets.Image(value=pil_bytes(result),   format="png")]),
]))

NameError: name 'dataset' is not defined